In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('accepted_2007_to_2018Q4.csv', low_memory=False)
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


First, I check de shape and column names of the dataset

In [3]:
df.shape

(2260701, 151)

In [4]:
df.columns

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade',
       ...
       'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
       'disbursement_method', 'debt_settlement_flag',
       'debt_settlement_flag_date', 'settlement_status', 'settlement_date',
       'settlement_amount', 'settlement_percentage', 'settlement_term'],
      dtype='object', length=151)

Then I only consider the states "Fully Paid", "Charged Off" or "Default" to create the dicotomic variable "Default". This reduces the rows by a little less than a million rows

In [5]:
# Keep only relevant loan statuses
valid_status = ["Fully Paid", "Charged Off", "Default"]

df = df[df["loan_status"].isin(valid_status)].copy()

# Create binary target
df["default"] = df["loan_status"].isin(["Charged Off", "Default"]).astype(int)

print(df["default"].value_counts(normalize=True))
print(df.shape)

0    0.80035
1    0.19965
Name: default, dtype: float64
(1345350, 152)


I eliminate all variables that are post loan, to avoid leakage (variables that contain information after default defeat the purpose of predicting said default)

In [6]:
leakage_cols = [
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d"
]

df = df.drop(columns=[col for col in leakage_cols if col in df.columns])

Now I whitlist the key variables to consider in the modeling. I don't consider grade or sub_grade, since this would be a decision taken after the borrower data is collected.

In [7]:
keep_cols = [
    # Loan characteristics
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "purpose",
    
    # Borrower profile
    "annual_inc",
    "emp_length",
    "home_ownership",
    
    # Credit history
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    
    # Time-based
    "issue_d",
    
    # Target
    "default"
]

df = df[[col for col in keep_cols if col in df.columns]].copy()

I check the new shape and the proportions of na values per variable

In [8]:
print(df.shape)
print(df.isna().mean().sort_values(ascending=False))

(1345350, 18)
emp_length        5.836102e-02
revol_util        6.370090e-04
dti               2.779946e-04
inq_last_6mths    7.433010e-07
loan_amnt         0.000000e+00
issue_d           0.000000e+00
total_acc         0.000000e+00
revol_bal         0.000000e+00
pub_rec           0.000000e+00
open_acc          0.000000e+00
delinq_2yrs       0.000000e+00
term              0.000000e+00
home_ownership    0.000000e+00
annual_inc        0.000000e+00
purpose           0.000000e+00
installment       0.000000e+00
int_rate          0.000000e+00
default           0.000000e+00
dtype: float64


I save the clean data

In [9]:
df.to_csv("lc_accepted_clean.csv", index=False)

print("Clean dataset saved as lc_accepted_clean.csv")

Clean dataset saved as lc_accepted_clean.csv


Create the metadata

In [10]:
import json

metadata = {
    "n_rows": int(df.shape[0]),
    "n_columns": int(df.shape[1]),
    "columns": list(df.columns),
    "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
    "default_rate": float(df["default"].mean()),
    "missing_rate": df.isna().mean().sort_values(ascending=False).to_dict()
}

with open("lc_accepted_clean_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata saved as lc_accepted_clean_metadata.json")

Metadata saved as lc_accepted_clean_metadata.json


And finally, a Sanity Cell.

In [11]:
assert "default" in df.columns
assert set(df["default"].unique()).issubset({0,1})

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Default rate:", round(df["default"].mean(), 4))

print("\nTop missing values:")
print(df.isna().mean().sort_values(ascending=False).head(10))

Rows: 1345350
Columns: 18
Default rate: 0.1996

Top missing values:
emp_length        5.836102e-02
revol_util        6.370090e-04
dti               2.779946e-04
inq_last_6mths    7.433010e-07
loan_amnt         0.000000e+00
issue_d           0.000000e+00
total_acc         0.000000e+00
revol_bal         0.000000e+00
pub_rec           0.000000e+00
open_acc          0.000000e+00
dtype: float64
